In [ ]:
import pandas as pd

data= pd.read_csv("../files/input/solicitudes_de_credito.csv", sep=";",encoding='utf-8')

In [51]:
solicitudes=data.copy()

In [86]:
solicitudes=solicitudes.drop('Unnamed: 0', axis=1)

In [87]:
cols = ["sexo", "tipo_de_emprendimiento", "idea_negocio","barrio","línea_credito"]
solicitudes[cols] = solicitudes[cols].apply(lambda x: x.str.lower())


In [88]:
import pandas as pd

def normalize_date_column(df, column_name):
    """
    Limpia y normaliza una columna de fechas en el DataFrame.
    Detecta formatos mixtos automáticamente (DD/MM/YYYY, YYYY-MM-DD, YYYY/MM/DD, etc.)
    y las convierte a un formato uniforme YYYY-MM-DD.
    """

    # Copiar y limpiar
    fechas = df[column_name].astype(str).str.strip()

    # Lista de formatos comunes que intentaremos
    formatos = [
        "%Y-%m-%d", "%Y/%m/%d",  # ISO
        "%d/%m/%Y", "%d-%m-%Y",  # Latino/europeo
    ]

    # Intentar cada formato hasta lograr la conversión
    fechas_convertidas = pd.to_datetime(fechas, errors="coerce", infer_datetime_format=True)

    # Completar las que quedaron NaT con los otros formatos posibles
    for fmt in formatos:
        mask = fechas_convertidas.isna()
        if mask.any():
            fechas_convertidas.loc[mask] = pd.to_datetime(
                fechas[mask], format=fmt, errors="coerce"
            )

    # Guardar en el DataFrame
    df[column_name] = fechas_convertidas.dt.strftime("%Y-%m-%d")

    return df

In [89]:
normalize_date_column(df=solicitudes, column_name="fecha_de_beneficio")

C:\Users\karla\AppData\Local\Temp\ipykernel_19476\667178894.py:20: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  fechas_convertidas = pd.to_datetime(fechas, errors="coerce", infer_datetime_format=True)


,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito,key,key1,key2
0,masculino,comercio,NaN,NaN,3,10.0,2018-07-13,5000000.0,NaN,"[de, fabrica]",[prado],[microempresarial]
1,femenino,servicio,NaN,NaN,1,9.0,2017-10-30,6000000.0,NaN,"[educativo, recreativo]","[barrio, caicedo]",[microempresarial]
2,femenino,comercio,NaN,NaN,2,4.0,2017-08-03,7300000.0,NaN,[tienda],[aranjuez],[microempresarial]
3,masculino,comercio,NaN,NaN,3,60.0,2017-11-20,7000000.0,NaN,[variedades],"[cabecera, cristobal, san]",[microempresarial]
4,femenino,comercio,NaN,NaN,2,7.0,2017-02-23,5000000.0,NaN,"[de, fabrica]",[robledo],[microempresarial]
...,...,...,...,...,...,...,...,...,...,...,...,...
10915,masculino,agropecuaria,NaN,NaN,1,8.0,2018-05-09,7000000.0,NaN,"[comidas, rapidas]",[villatina],[microempresarial]
10916,masculino,comercio,NaN,NaN,3,8.0,2017-01-23,7377000.0,NaN,"[arepas, de, fabrica]","[hermosa, villa]",[microempresarial]
10917,femenino,comercio,NaN,NaN,1,60.0,2018-04-12,7800000.0,NaN,[variedades],"[cabecera, cristobal, san]",[microempresarial]
10918,femenino,servicio,NaN,NaN,2,3.0,2017-06-01,7370000.0,NaN,"[cafe, e, internet]","[manrique, oriental]",[microempresarial]


In [94]:
def create_normalized_key(df, column_name, new_column="key"):
    """
    Crea una nueva columna en el DataFrame que contenga una versión normalizada
    de la columna especificada, eliminando tildes, puntuación y palabras auxiliares
    en español, sin unir indebidamente palabras y aplicando stemming.
    """

    # Copia de la columna original
    df[new_column] = df[column_name].astype(str).str.lower()

    # 🔹 Reemplazar separadores por espacio (evita pegar palabras)
    df[new_column] = df[new_column].str.replace(r"[_\-]+", " ", regex=True)

    # 🔹 Quitar signos de puntuación
    df[new_column] = df[new_column].str.replace(r"[^\w\s]", " ", regex=True)

        # 🔹 Tokenizar
    df[new_column] = df[new_column].str.strip()

    # 🔹 Eliminar duplicados y ordenar
    df[new_column] = df[new_column].apply(lambda x: sorted(set(x)))
    
    return df



In [95]:
create_normalized_key(df=solicitudes, column_name="idea_negocio", new_column="key")
solicitudes['key'].value_counts()

key
[a, n]    10920
Name: count, dtype: int64

In [77]:
create_normalized_key(df=solicitudes, column_name="barrio", new_column="key1")
solicitudes['key1'].value_counts()

key1
[robledo]                     1036
[1, central, manrique, no]     512
[1, javier, no, san]           454
[aranjuez]                     406
[aires, buenos]                399
                              ... 
[flora, villa]                   1
[del, doce, mirador]             1
[barro, suburbano]               1
[el, estadio]                    1
[el, llano]                      1
Name: count, Length: 224, dtype: int64

In [78]:
create_normalized_key(df=solicitudes, column_name="línea_credito", new_column="key2")
solicitudes['key2'].value_counts()

key2
[microempresarial]             10722
[ed, empresarial]                 73
[agropecuaria]                    61
[cap, juridica, semilla, y]       34
[credioportuno]                   21
[agropecuario, fomento]            5
[diaria, soli]                     2
[solidaria]                        1
[ayacucho, formal]                 1
Name: count, dtype: int64

In [79]:
solicitudes['idea_negocio']=solicitudes['key']
solicitudes['barrio']=solicitudes['key1']
solicitudes['línea_credito']=solicitudes['key2']

In [80]:
solicitudes.drop(columns=['key','key1','key2'])

,Unnamed: 0,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito
0,0,masculino,comercio,"[de, fabrica]",[prado],3,10.0,2018-07-13,5000000.0,[microempresarial]
1,1,femenino,servicio,"[educativo, recreativo]","[barrio, caicedo]",1,9.0,2017-10-30,6000000.0,[microempresarial]
2,2,femenino,comercio,[tienda],[aranjuez],2,4.0,2017-08-03,7300000.0,[microempresarial]
3,3,masculino,comercio,[variedades],"[cabecera, cristobal, san]",3,60.0,2017-11-20,7000000.0,[microempresarial]
4,4,femenino,comercio,"[de, fabrica]",[robledo],2,7.0,2017-02-23,5000000.0,[microempresarial]
...,...,...,...,...,...,...,...,...,...,...
10915,10915,masculino,agropecuaria,"[comidas, rapidas]",[villatina],1,8.0,2018-05-09,7000000.0,[microempresarial]
10916,10916,masculino,comercio,"[arepas, de, fabrica]","[hermosa, villa]",3,8.0,2017-01-23,7377000.0,[microempresarial]
10917,10917,femenino,comercio,[variedades],"[cabecera, cristobal, san]",1,60.0,2018-04-12,7800000.0,[microempresarial]
10918,10918,femenino,servicio,"[cafe, e, internet]","[manrique, oriental]",2,3.0,2017-06-01,7370000.0,[microempresarial]


In [53]:
def normalizar_fecha(texto):
        try:
            if re.match(r'^\d{4}/\d{1,2}/\d{1,2}$', str(texto)):
                partes = texto.split('/')
                año = partes[0]
                mes = partes[1].zfill(2)
                dia = partes[2].zfill(2)
                return f"{dia}/{mes}/{año}"
            else:
                return texto
        except:
            return texto

solicitudes['fecha_de_beneficio'] = solicitudes['fecha_de_beneficio'].apply(normalizar_fecha)
solicitudes['fecha_de_beneficio'] = pd.to_datetime(solicitudes['fecha_de_beneficio'], errors='coerce', dayfirst=True)

In [54]:
solicitudes['fecha_de_beneficio'].value_counts()

fecha_de_beneficio
2018-10-03    69
2018-10-11    62
2018-09-14    43
2018-10-25    42
2018-04-25    42
              ..
2016-01-05     1
2017-08-05     1
2016-07-31     1
2019-01-02     1
2016-01-27     1
Name: count, Length: 795, dtype: int64

In [83]:
solicitudes['fecha_de_beneficio'].value_counts()

fecha_de_beneficio
2018-10-03    69
2018-10-11    62
2018-09-14    43
2018-10-25    42
2018-04-25    42
              ..
2016-01-05     1
2017-08-05     1
2016-07-31     1
2019-01-02     1
2016-01-27     1
Name: count, Length: 795, dtype: int64

In [84]:
solicitudes.isna().sum()

Unnamed: 0                  0
sexo                        0
tipo_de_emprendimiento    102
idea_negocio                0
barrio                      0
estrato                     0
comuna_ciudadano            0
fecha_de_beneficio          0
monto_del_credito         557
línea_credito               0
key                         0
key1                        0
key2                        0
dtype: int64

In [63]:
 # Arreglar columna monto
solicitudes['monto_del_credito'] = solicitudes['monto_del_credito'].astype(str)
solicitudes['monto_del_credito'] = solicitudes['monto_del_credito'].str.replace('$', '').str.replace('_', ' ').str.replace('-', ' ')
solicitudes['monto_del_credito'] = solicitudes['monto_del_credito'].str.replace(' ', '') 
solicitudes['monto_del_credito'] = solicitudes['monto_del_credito'].str.replace(',', '') 
solicitudes['monto_del_credito'] = solicitudes['monto_del_credito'].astype(float)

In [96]:
import pandas as pd

def normalize_amount_column(df, column_name):
    """
    Limpia y normaliza una columna de montos.
    - Elimina símbolos de moneda, comas, espacios y puntos innecesarios.
    - Convierte los valores a tipo numérico (float).
    """

    # Convertir todo a string y limpiar caracteres no numéricos
    df[column_name] = (
        df[column_name]
        .astype(str)
        .replace({'$':'','_':' ','-':' '}, regex=True)  # deja solo números y separadores
        .str.replace(",", "", regex=True)           # quita comas de miles
    )

    # Convertir a número (NaN si no se puede)
    df[column_name] = df[column_name].astype(float)

    return df


In [97]:
normalize_amount_column(solicitudes, "monto_del_credito")


,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito,key,key1,key2
0,masculino,comercio,NaN,NaN,3,10.0,2018-07-13,5000000.0,NaN,"[a, n]",[prado],[microempresarial]
1,femenino,servicio,NaN,NaN,1,9.0,2017-10-30,6000000.0,NaN,"[a, n]","[barrio, caicedo]",[microempresarial]
2,femenino,comercio,NaN,NaN,2,4.0,2017-08-03,7300000.0,NaN,"[a, n]",[aranjuez],[microempresarial]
3,masculino,comercio,NaN,NaN,3,60.0,2017-11-20,7000000.0,NaN,"[a, n]","[cabecera, cristobal, san]",[microempresarial]
4,femenino,comercio,NaN,NaN,2,7.0,2017-02-23,5000000.0,NaN,"[a, n]",[robledo],[microempresarial]
...,...,...,...,...,...,...,...,...,...,...,...,...
10915,masculino,agropecuaria,NaN,NaN,1,8.0,2018-05-09,7000000.0,NaN,"[a, n]",[villatina],[microempresarial]
10916,masculino,comercio,NaN,NaN,3,8.0,2017-01-23,7377000.0,NaN,"[a, n]","[hermosa, villa]",[microempresarial]
10917,femenino,comercio,NaN,NaN,1,60.0,2018-04-12,7800000.0,NaN,"[a, n]","[cabecera, cristobal, san]",[microempresarial]
10918,femenino,servicio,NaN,NaN,2,3.0,2017-06-01,7370000.0,NaN,"[a, n]","[manrique, oriental]",[microempresarial]


In [100]:
solicitudes['monto_del_credito'].value_counts()

monto_del_credito
7800000.0     1190
7000000.0     1068
5000000.0     1056
6000000.0      961
4000000.0      769
              ... 
22131510.0       1
7320000.0        1
4850000.0        1
8261160.0        1
80000000.0       1
Name: count, Length: 270, dtype: int64

In [98]:
print(len(solicitudes["monto_del_credito"].unique()))

271


In [ ]:
solicitudes=solicitudes.drop_duplicates()

In [ ]:
solicitudes=solicitudes.dropna()

In [ ]:
solicitudes["línea_credito"].value_counts()